In [1]:
import pandas as pd

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

train.head()

Train Shape: (891, 12)
Test Shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [3]:
train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [4]:
train["Age"] = train["Age"].fillna(train["Age"].median())

train["Age"].isnull().sum()

np.int64(0)

In [5]:
train["Embarked"] = train["Embarked"].fillna("S")

train["Embarked"].isnull().sum()

np.int64(0)

In [6]:
train["Cabin"].isnull().sum()

np.int64(687)

In [7]:
train = train.drop(columns=["Cabin"])

train.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Embarked'],
      dtype='str')

In [8]:
train["Survived"].value_counts()

Survived
0    549
1    342
Name: count, dtype: int64

In [9]:
train["Sex"] = train["Sex"].map({
    "male": 0,
    "female": 1
})

train["Sex"].head()

0    0
1    1
2    1
3    1
4    0
Name: Sex, dtype: int64

In [10]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

X = train[features]

y = train["Survived"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (891, 6)
y shape: (891,)


In [11]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X, y)

print("Model trained successfully!")

Model trained successfully!


In [12]:
test["Age"] = test["Age"].fillna(
    test["Age"].median()
)

test["Fare"] = test["Fare"].fillna(
    test["Fare"].median()
)

test["Sex"] = test["Sex"].map({
    "male": 0,
    "female": 1
})

print("Test data prepared!")

Test data prepared!


In [13]:
predictions = model.predict(test[features])

print(predictions[:20])

[0 0 1 1 0 0 0 0 1 0 0 0 1 0 1 1 0 1 0 0]


In [14]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": predictions
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created!")

submission.csv created!


In [15]:
train[features].head()

,Pclass,Sex,Age,SibSp,Parch,Fare
0,3,0,22.0,1,0,7.2500
1,1,1,38.0,1,0,71.2833
2,3,1,26.0,0,0,7.9250
3,1,1,35.0,1,0,53.1000
4,3,0,35.0,0,0,8.0500


In [16]:
submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,1
3,895,1
4,896,0


In [17]:
submission["Survived"].value_counts()

Survived
0    264
1    154
Name: count, dtype: int64

In [18]:
from sklearn.metrics import accuracy_score

train_predictions = model.predict(X)

accuracy = accuracy_score(y, train_predictions)

print("Training Accuracy:", accuracy)

Training Accuracy: 0.9797979797979798


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_valid.shape)

(712, 6)
(179, 6)


In [20]:
from sklearn.ensemble import RandomForestClassifier

model2 = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model2.fit(X_train, y_train)

print("New model trained!")

New model trained!


In [21]:
from sklearn.metrics import accuracy_score

valid_predictions = model2.predict(X_valid)

validation_accuracy = accuracy_score(
    y_valid,
    valid_predictions
)

print("Validation Accuracy:", validation_accuracy)

Validation Accuracy: 0.8044692737430168


In [22]:
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1

train["FamilySize"].head()

0    2
1    2
2    1
3    2
4    1
Name: FamilySize, dtype: int64

In [23]:
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

test["FamilySize"].head()

0    1
1    2
2    1
3    1
4    3
Name: FamilySize, dtype: int64

In [24]:
features_v2 = [
    "Pclass",
    "Sex",
    "Age",
    "Fare",
    "FamilySize"
]

X2 = train[features_v2]
y2 = train["Survived"]

print(X2.shape)

(891, 5)


In [25]:
from sklearn.model_selection import train_test_split

X2_train, X2_valid, y2_train, y2_valid = train_test_split(
    X2,
    y2,
    test_size=0.2,
    random_state=42
)

print(X2_train.shape)
print(X2_valid.shape)

(712, 5)
(179, 5)


In [26]:
from sklearn.ensemble import RandomForestClassifier

model_v2 = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5
)

model_v2.fit(X2_train, y2_train)

print("Model v2 trained!")

Model v2 trained!


In [27]:
from sklearn.metrics import accuracy_score

v2_predictions = model_v2.predict(X2_valid)

v2_accuracy = accuracy_score(
    y2_valid,
    v2_predictions
)

print("Model v2 Validation Accuracy:", v2_accuracy)

Model v2 Validation Accuracy: 0.7988826815642458


In [28]:
train["Embarked"].value_counts()

Embarked
S    646
C    168
Q     77
Name: count, dtype: int64

In [29]:
train["Embarked"] = train["Embarked"].map({
    "S": 0,
    "C": 1,
    "Q": 2
})

test["Embarked"] = test["Embarked"].fillna("S")

test["Embarked"] = test["Embarked"].map({
    "S": 0,
    "C": 1,
    "Q": 2
})

print(train["Embarked"].head())

0    0
1    1
2    0
3    0
4    0
Name: Embarked, dtype: int64


In [30]:
features_v3 = [
    "Pclass",
    "Sex",
    "Age",
    "Fare",
    "Embarked",
    "FamilySize"
]

X3 = train[features_v3]
y3 = train["Survived"]

print(X3.shape)

(891, 6)


In [31]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X3_train, X3_valid, y3_train, y3_valid = train_test_split(
    X3,
    y3,
    test_size=0.2,
    random_state=42
)

model_v3 = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5
)

model_v3.fit(X3_train, y3_train)

v3_predictions = model_v3.predict(X3_valid)

v3_accuracy = accuracy_score(
    y3_valid,
    v3_predictions
)

print("Model v3 Validation Accuracy:", v3_accuracy)

Model v3 Validation Accuracy: 0.8100558659217877


In [32]:
model_final = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5
)

model_final.fit(X3, y3)

print("Final model trained on all data!")

Final model trained on all data!


In [33]:
final_predictions = model_final.predict(
    test[features_v3]
)

submission_v2 = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": final_predictions
})

submission_v2.to_csv(
    "submission_v2.csv",
    index=False
)

print("submission_v2.csv created!")

submission_v2.csv created!


In [34]:
submission_v2.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
